## Question: Do spatially nearby thalamocortical axons target similar cortical neurons or participate in similar feed-forward inhibitory motifs?

In [7]:
from pathlib import Path
import os
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from scipy.spatial.distance import pdist, squareform

sns.set_theme(style="whitegrid", context="talk")

DATA_DIR = Path(".")
RANDOM_SEED = 0
N_PERMUTATIONS = 1000
N_DISTANCE_BINS = 5

# Example ctr_pt_position values (~[665051, 653062, 511560]) are in nm.
# Allowed values: "nm", "um", or "voxel".
COORDINATE_UNIT = "nm"
VOXEL_SIZE_UM = np.array([0.008, 0.008, 0.045])  # used only for unit="voxel"

# Use the synapse-center coordinate column explicitly.
POSITION_COLUMN = "ctr_pt_position"

In [2]:
thal_syn = pd.read_csv(DATA_DIR / "thal_syn.csv")
I_syn = pd.read_csv(DATA_DIR / "I_syn.csv")

cell_type_path = DATA_DIR / "cell_types.csv"
if cell_type_path.exists():
    cell_types = pd.read_csv(cell_type_path)
else:
    from caveclient import CAVEclient

    if "API_SECRET" not in os.environ:
        raise RuntimeError(
            "Provide cell_types.csv or set API_SECRET before running this cell."
        )
    client = CAVEclient(
        "v1dd_public",
        server_address="https://global.em.brain.allentech.org",
        auth_token=os.environ["API_SECRET"],
    )
    cell_types = client.materialize.query_table("cell_type_multifeature_v1")

ct = cell_types[["pt_root_id", "classification_system", "cell_type"]].drop_duplicates("pt_root_id")

print(f"Thalamocortical axons: {thal_syn['pre_pt_root_id'].nunique():,}")
print(f"Thalamocortical synapses: {len(thal_syn):,}")
print(f"Inhibitory-output synapses: {len(I_syn):,}")

Thalamocortical axons: 97
Thalamocortical synapses: 463,661
Inhibitory-output synapses: 2,045,667


In [3]:
thal_syn

,id,created,superceded_id,valid,size,pre_pt_supervoxel_id,pre_pt_root_id,post_pt_supervoxel_id,post_pt_root_id,pre_pt_position,post_pt_position,ctr_pt_position
0,282681478,2021-11-19 01:35:28.272045+00:00,NaN,True,2902.0,88397162936966358,864691132647486796,88397162936965005,864691132832053252,[665158 653081 511605],[665332 652761 511380],[665051 653062 511560]
1,324977225,2021-11-18 22:16:58.141699+00:00,NaN,True,739.0,89662287497449724,864691132804540262,89591918753323750,864691132612319611,[709739 598771 182340],[709341 598781 182655],[709555 598810 182430]
2,404535272,2021-11-18 22:03:58.345906+00:00,NaN,True,750.0,93321462999446267,864691132647486796,93321462999447466,864691131718083940,[840058 599654 438525],[839622 599624 438525],[839816 599692 438435]
3,417826507,2021-11-19 02:10:08.767039+00:00,NaN,True,1808.0,93532774786769330,864691132843175686,93532774786771710,864691132612319611,[846732 606385 243180],[847130 606638 243405],[846926 606531 243270]
4,295785734,2021-11-18 22:51:18.353664+00:00,NaN,True,2390.0,88465400438245810,864691132629122108,88535769182411361,864691132022324186,[669610 575520 205425],[670105 575520 205740],[669911 575491 205650]
...,...,...,...,...,...,...,...,...,...,...,...,...
463656,432097798,2021-11-19 00:40:18.412715+00:00,NaN,True,5283.0,94519449503331350,864691132731716336,94519449503338875,864691132842860980,[881875 659425 397890],[882205 659803 398070],[882215 659570 398160]
463657,400168853,2021-11-19 00:27:55.774292+00:00,NaN,True,2630.0,92620042897998710,864691132731716336,92619974178505838,864691132644530230,[814615 679252 309690],[814266 679067 310185],[814305 679242 309960]
463658,307436428,2021-11-19 04:45:30.802793+00:00,NaN,True,903.0,89168607246386510,864691132549572162,89168607246380012,864691132609963587,[694083 557720 345555],[694345 557730 345285],[694044 557604 345330]
463659,376398490,2021-11-19 01:09:06.746304+00:00,NaN,True,1563.0,91980127265041365,864691132836641685,91980127265035005,864691132596158271,[793207 441117 362205],[793246 440865 361800],[793508 440932 362025]


In [4]:
def count_matrix(df, row="pre_pt_root_id", col="post_pt_root_id"):
    counts = df.groupby([row, col]).size().rename("n_synapses").reset_index()
    return counts.pivot(index=row, columns=col, values="n_synapses").fillna(0)

thal_typed = thal_syn.merge(
    ct, left_on="post_pt_root_id", right_on="pt_root_id", how="inner"
)
T_E_syn = thal_typed.loc[thal_typed["classification_system"] == "excitatory"].copy()
T_I_syn = thal_typed.loc[thal_typed["classification_system"] == "inhibitory"].copy()

I_syn_typed = I_syn.merge(
    ct, left_on="post_pt_root_id", right_on="pt_root_id", how="inner"
)
I_E_syn = I_syn_typed.loc[I_syn_typed["classification_system"] == "excitatory"].copy()

W_TE = count_matrix(T_E_syn)
W_TI = count_matrix(T_I_syn)
W_IE = count_matrix(I_E_syn)

common_I = W_TI.columns.intersection(W_IE.index)
A_TI = W_TI.loc[:, common_I].gt(0).astype(np.int32)
A_IE = W_IE.loc[common_I, :].gt(0).astype(np.int32)
FFI_paths = A_TI @ A_IE

# Bias-safe alignment: do not intersect W_TE columns with FFI_paths columns.
Direct = W_TE.copy()
FFI = FFI_paths.reindex(index=Direct.index, columns=Direct.columns, fill_value=0)
TI = W_TI.reindex(index=Direct.index, fill_value=0)

assert Direct.index.equals(FFI.index)
assert Direct.columns.equals(FFI.columns)

print("Direct T × E:", Direct.shape)
print("T × I:", TI.shape)
print("FFI T × E:", FFI.shape)
print(
    "E neurons absent from raw FFI_paths and zero-filled:",
    len(W_TE.columns.difference(FFI_paths.columns)),
)

Direct T × E: (97, 17397)
T × I: (97, 2752)
FFI T × E: (97, 17397)
E neurons absent from raw FFI_paths and zero-filled: 1427


In [8]:
def parse_position(value):
    if isinstance(value, str):
        # Handles both '[665051 653062 511560]' and comma-separated strings.
        number_pattern = r"[-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?"
        arr = np.asarray(re.findall(number_pattern, value), dtype=float)
    else:
        try:
            arr = np.asarray(value, dtype=float).reshape(-1)
        except (TypeError, ValueError):
            return np.array([np.nan, np.nan, np.nan])
    if arr.size < 3:
        return np.array([np.nan, np.nan, np.nan])
    return arr[:3].astype(float)

def extract_xyz(df, position_column=None):
    if position_column is not None:
        if position_column not in df.columns:
            raise KeyError(
                f"Requested position column '{position_column}' was not found. "
                f"Available columns: {df.columns.tolist()}"
            )
        xyz = np.vstack(df[position_column].map(parse_position).to_numpy())
        return xyz, position_column

    triplets = [
        ("ctr_pt_x", "ctr_pt_y", "ctr_pt_z"),
        ("x", "y", "z"),
    ]
    for cols in triplets:
        if set(cols).issubset(df.columns):
            return df.loc[:, cols].astype(float).to_numpy(), cols

    candidates = ["ctr_pt_position", "pre_pt_position", "post_pt_position"]
    for col in candidates:
        if col is not None and col in df.columns:
            xyz = np.vstack(df[col].map(parse_position).to_numpy())
            return xyz, col

    raise KeyError(
        "No coordinate columns found. Set POSITION_COLUMN or add x/y/z columns. "
        f"Available columns: {df.columns.tolist()}"
    )

xyz, position_source = extract_xyz(thal_syn, POSITION_COLUMN)
if COORDINATE_UNIT == "nm":
    xyz = xyz * 1e-3
elif COORDINATE_UNIT == "voxel":
    xyz = xyz * VOXEL_SIZE_UM
elif COORDINATE_UNIT != "um":
    raise ValueError("COORDINATE_UNIT must be 'nm', 'um', or 'voxel'.")

synapse_xyz = pd.DataFrame(xyz, columns=["x_um", "y_um", "z_um"], index=thal_syn.index)
synapse_xyz["thal_id"] = thal_syn["pre_pt_root_id"].to_numpy()
synapse_xyz = synapse_xyz.dropna(subset=["x_um", "y_um", "z_um"])

axon_centroids = synapse_xyz.groupby("thal_id")[["x_um", "y_um", "z_um"]].mean()
axon_ids = Direct.index.intersection(axon_centroids.index)

Direct = Direct.loc[axon_ids]
TI = TI.loc[axon_ids]
FFI = FFI.loc[axon_ids]
axon_centroids = axon_centroids.loc[axon_ids]

print("Position source:", position_source)
print("Input coordinate unit:", COORDINATE_UNIT, "→ analysis unit: µm")
print(f"Axons with valid centroids: {len(axon_ids):,} / {len(W_TE.index):,}")
axon_centroids.head()

Position source: ctr_pt_position
Input coordinate unit: nm → analysis unit: µm
Axons with valid centroids: 97 / 97


,x_um,y_um,z_um
864691132537321562,933.026001,664.391215,221.485082
864691132549572162,786.099424,619.866593,303.357535
864691132572564252,694.872652,588.115512,379.825932
864691132582739474,917.519633,649.441007,247.516352
864691132582742802,1020.457343,711.424245,233.629074


In [1]:
def binary_jaccard_matrix(matrix):
    X = np.asarray(matrix, dtype=np.int32) > 0
    X = X.astype(np.int32)
    intersection = X @ X.T
    degree = X.sum(axis=1)
    union = degree[:, None] + degree[None, :] - intersection
    with np.errstate(divide="ignore", invalid="ignore"):
        similarity = intersection / union
    similarity[union == 0] = np.nan
    np.fill_diagonal(similarity, 1.0)
    return similarity

def safe_spearman(x, y):
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.unique(x[valid]).size < 2 or np.unique(y[valid]).size < 2:
        return np.nan
    return stats.spearmanr(x[valid], y[valid]).statistic

centroid_distance = squareform(pdist(axon_centroids.to_numpy(), metric="euclidean"))
similarity_matrices = {
    "Direct E targets": binary_jaccard_matrix(Direct),
    "Shared I partners": binary_jaccard_matrix(TI),
    "FFI E endpoints": binary_jaccard_matrix(FFI),
}

upper = np.triu_indices(len(axon_ids), k=1)
pair_df = pd.DataFrame({
    "axon_a": np.asarray(axon_ids)[upper[0]],
    "axon_b": np.asarray(axon_ids)[upper[1]],
    "distance_um": centroid_distance[upper],
})
for name, matrix in similarity_matrices.items():
    pair_df[name] = matrix[upper]

print(f"Axon pairs: {len(pair_df):,}")
pair_df.head()

NameError: name 'squareform' is not defined